# Climate incentives and biodiversity complements: Full reproduction

Recalculate the numerical evidence from the original European coefficients for 23 EU countries. The analysis verifies the policy certificates and regenerates the tables and five figures. Select **Runtime > Run all** to download `outputs.zip`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, version
from IPython.display import display, Markdown, FileLink

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "code/reproduce.py").is_file() and (p / "inputs/SOURCE.json").is_file()), None)
if root is None:
    root = Path(tempfile.mkdtemp(prefix="climate_biodiversity_")) / "climate-biodiversity-complements"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/calinadriancomes/climate-biodiversity-complements.git",
                    str(root)], check=True, capture_output=True, text=True)
print("Repository:", root.name)


In [ ]:
required = dict(line.strip().split("==") for line in (root / "requirements.txt").read_text().splitlines()
                if line.strip() and not line.lstrip().startswith("#"))

def needs_install(name, expected):
    try:
        return version(name) != expected
    except PackageNotFoundError:
        return True

if any(needs_install(name, expected) for name, expected in required.items()):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(root / "requirements.txt")], check=True)
assert all(version(name) == expected for name, expected in required.items())


In [ ]:
result = subprocess.run(
    [sys.executable, "-B", str(root / "code/reproduce.py"), "--full"],
    cwd=root, capture_output=True, text=True)
if result.returncode:
    print(result.stdout[-4000:])
    print(result.stderr[-4000:])
    result.check_returncode()


In [ ]:
out = root / "outputs"
verified = json.loads((out / "matrix_replay/verification.json").read_text())
safeguard = json.loads((out / "safeguard/independent_verification.json").read_text())
assert verified["status"] == "PASS" and safeguard["status"] == "verified"
display(Markdown((out / "tables/Climate_coefficient_frontier.md").read_text()))
archive = root / "outputs.zip"
assert archive.is_file()
print("Full source-coefficient verification: PASS")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(archive))
else:
    display(FileLink(str(archive)))
